# Einflussanalyse der Kundenzufriedenheit

In [1]:
import sys
import os

sys.path.append(
    os.path.abspath(
        os.path.join(os.getcwd(), "..")
    )
)

import warnings
warnings.filterwarnings("ignore")

# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy.stats import (
    ttest_ind,
    f_oneway,
    chi2_contingency,
    pearsonr,
    spearmanr
)

# Machine Learning
from sklearn.preprocessing import LabelEncoder

# Database Loader
from db.data_loader import load_data

# Plot Settings
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

1. LOAD DATA UND DATA OVERVIEW

In [2]:
from db.data_loader import load_data

df = load_data()

df.head()

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.info()
df.shape

Rows: 1200
Columns: 13
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   datum                1200 non-null   str    
 1   marke                1200 non-null   str    
 2   modell               1200 non-null   str    
 3   preis_euro           1200 non-null   int64  
 4   verkaufszahl         1200 non-null   int64  
 5   kraftstoff           1200 non-null   str    
 6   getriebe             1200 non-null   str    
 7   hubraum_l            1200 non-null   float64
 8   bundesland           1200 non-null   str    
 9   kundenzufriedenheit  1200 non-null   float64
 10  jahr                 1200 non-null   int64  
 11  monat                1200 non-null   int64  
 12  wochentag            1200 non-null   str    
dtypes: float64(2), int64(4), str(7)
memory usage: 122.0 KB


(1200, 13)

Numerische und Kategorische Variablen 

In [3]:
numerical_columns = [
    "preis_euro",
    "verkaufszahl",
    "hubraum_l",
    "kundenzufriedenheit",
    "jahr",
    "monat"
]

categorical_columns = [
    "marke",
    "modell",
    "kraftstoff",
    "getriebe",
    "bundesland",
    "wochentag"
]

print(numerical_columns)
print(categorical_columns)

['preis_euro', 'verkaufszahl', 'hubraum_l', 'kundenzufriedenheit', 'jahr', 'monat']
['marke', 'modell', 'kraftstoff', 'getriebe', 'bundesland', 'wochentag']


2. EINFLUSSANALYSE

2.1  Numerische Variablen → Kundenzufriedenheit  
Spearman  

In [4]:
from scipy.stats import spearmanr
import pandas as pd

numerical_features = [
    "preis_euro",
    "verkaufszahl",
    "hubraum_l",
    "jahr",
    "monat"
]

results_num = []

for col in numerical_features:

    rho, p = spearmanr(
        df[col],
        df["kundenzufriedenheit"]
    )

    results_num.append({
        "Variable": col,
        "Spearman_rho": rho,
        "p_value": p
    })

numerical_importance = pd.DataFrame(results_num)

numerical_importance = (
    numerical_importance
    .sort_values(
        by="Spearman_rho",
        key=abs,
        ascending=False
    )
)

numerical_importance

,Variable,Spearman_rho,p_value
1,verkaufszahl,-0.076122,0.008339
3,jahr,0.047233,0.101966
4,monat,0.044873,0.120278
0,preis_euro,0.020314,0.482032
2,hubraum_l,-0.016723,0.562758


2.2 Kategoriale Variablen → Kundenzufriedenheit  
ANOVA  
η² (Eta Squared)  

In [5]:
from scipy.stats import f_oneway
import numpy as np

def anova_eta_squared(df, cat_col, target):

    groups = [
        group[target].values
        for _, group in df.groupby(cat_col)
    ]

    f_stat, p_value = f_oneway(*groups)

    grand_mean = df[target].mean()

    ss_between = sum(
        len(group) *
        (group.mean() - grand_mean) ** 2
        for _, group in df.groupby(cat_col)[target]
    )

    ss_total = sum(
        (df[target] - grand_mean) ** 2
    )

    eta_sq = ss_between / ss_total

    return f_stat, p_value, eta_sq

In [6]:
categorical_features = [
    "marke",
    "modell",
    "kraftstoff",
    "getriebe",
    "bundesland",
    "wochentag"
]

results_cat = []

for col in categorical_features:

    f, p, eta = anova_eta_squared(
        df,
        col,
        "kundenzufriedenheit"
    )

    results_cat.append({
        "Variable": col,
        "F": f,
        "p_value": p,
        "eta_squared": eta
    })

categorical_importance = pd.DataFrame(results_cat)

categorical_importance = (
    categorical_importance
    .sort_values(
        by="eta_squared",
        ascending=False
    )
)

categorical_importance

,Variable,F,p_value,eta_squared
1,modell,0.948710,0.521256,0.015046
4,bundesland,1.628322,0.149524,0.006773
3,getriebe,3.885251,0.048942,0.003233
5,wochentag,0.587006,0.740942,0.002944
0,marke,0.313779,0.868878,0.001049
2,kraftstoff,0.286018,0.835524,0.000717


Tabelle 1. Einfluss numerischer Variablen auf die Kundenzufriedenheit (Spearman)  
Variable	Spearman ρ	p-Wert	Einfluss  
verkaufszahl	-0.076	0.008	sehr schwach, statistisch signifikant  
jahr	0.047	0.102	kein signifikanter Zusammenhang  
monat	0.045	0.120	kein signifikanter Zusammenhang  
preis_euro	0.020	0.482	kein signifikanter Zusammenhang  
hubraum_l	-0.017	0.563	kein signifikanter Zusammenhang  
  
  Interpretation auf Deutsch  
  
Die Rangkorrelationsanalyse nach Spearman zeigt insgesamt nur sehr geringe Korrelationskoeffizienten (|ρ| < 0,1). Lediglich die Verkaufszahl weist einen statistisch signifikanten, jedoch praktisch vernachlässigbaren negativen Zusammenhang mit der Kundenzufriedenheit auf (ρ = -0,076; p = 0,008). Für Preis, Hubraum, Jahr und Monat konnten keine signifikanten Zusammenhänge festgestellt werden.  
  
  Tabelle 2. Einfluss kategorialer Variablen auf die Kundenzufriedenheit (ANOVA)  
Variable	p-Wert	η²	Einfluss  
modell	0.521	0.015	schwach  
bundesland	0.150	0.007	kein signifikanter Einfluss  
getriebe	0.049	0.003	statistisch signifikant, praktisch kein Einfluss  
wochentag	0.741	0.003	kein signifikanter Einfluss  
marke	0.869	0.001	kein signifikanter Einfluss  
kraftstoff	0.836	0.001	kein signifikanter Einfluss  
  
  Interpretation auf Deutsch  

Die Varianzanalyse zeigt insgesamt sehr geringe Effektstärken. Lediglich die Getriebeart weist einen statistisch signifikanten Zusammenhang mit der Kundenzufriedenheit auf (p = 0,049), die Effektstärke ist jedoch mit η² = 0,003 praktisch vernachlässigbar. Für Marke, Modell, Kraftstoff, Bundesland und Wochentag konnten keine relevanten Einflüsse auf die Kundenzufriedenheit festgestellt werden.  
  
  Gesamtfazit (Deutsch)  

Im Gegensatz zur Preisvorhersage konnten für die Zielvariable „Kundenzufriedenheit“ keine starken Einflussfaktoren identifiziert werden. Sowohl die Korrelationsanalyse als auch die ANOVA zeigen überwiegend sehr geringe Effektstärken und fehlende statistisch relevante Zusammenhänge. Die vorhandenen Merkmale erklären die Kundenzufriedenheit daher nur unzureichend. Es ist zu erwarten, dass Machine-Learning-Modelle für diese Zielvariable nur eine begrenzte Vorhersagegüte erreichen.  